# socbench results explorer

Load the artifacts of a completed run and slice them by **stratum**,
**persona**, and **provider**. Works on any `runs/<run_id>/` directory (a
local mock run from `quickstart.ipynb` or a published run downloaded from a
results bucket).

Reads three artifacts:

- `summary.json`: run-level `scoring` + `cache` + latency rollups.
- `eval_units_summary.jsonl`: one scored row per `(eval_unit, persona,
  provider)`, with the three lens F1s, `gold_label`, `verdict`, `defect`.
- `run_metadata.json`: manifest SHAs, ablation tag, pricing snapshot.

If `ablations/<dataset_hash>/<seed>/ablation_summary.json` exists, the final
section renders the ablation deltas too.

In [ ]:
import json
from pathlib import Path

import pandas as pd

# Locate the repo root.
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent

# RUN_DIR: set explicitly, or leave as None to auto-pick the most recent run.
RUN_DIR: Path | None = None

if RUN_DIR is None:
    runs_root = REPO / "runs"
    candidates = sorted(p for p in runs_root.glob("*") if (p / "summary.json").exists())
    if not candidates:
        raise FileNotFoundError(
            "no runs found under runs/. Run quickstart.ipynb first, "
            "or set RUN_DIR to a specific run directory."
        )
    RUN_DIR = candidates[-1]  # run_id is timestamp-prefixed → newest last

RUN_DIR = Path(RUN_DIR)
print("exploring:", RUN_DIR.relative_to(REPO))

In [ ]:
summary = json.loads((RUN_DIR / "summary.json").read_text())
metadata = json.loads((RUN_DIR / "run_metadata.json").read_text())

units = [
    json.loads(line)
    for line in (RUN_DIR / "eval_units_summary.jsonl").read_text().splitlines()
    if line.strip()
]
units_df = pd.DataFrame(units)

print(f"run_id:    {metadata['run_id']}")
print(f"ablation:  {metadata['ablation']}   mode: {metadata['mode']}")
print(f"dataset:   {metadata['dataset_hash']}   seed: {metadata['sample_seed']}")
print(f"manifests: prompts={metadata['prompts_manifest_sha'][:8]} "
      f"playbooks={metadata['playbooks_manifest_sha'][:8]} "
      f"tools={metadata['tools_manifest_sha'][:8]}")
print(f"pricing snapshot: {metadata['pricing_snapshot_date']}")
print(f"\n{len(units_df)} scored renderings")
units_df.head()

## Scoring by provider × persona

The run-level `scoring` block, as a table.

In [ ]:
scoring_df = pd.DataFrame(summary["scoring"]).T
scoring_df.index.name = "provider/persona"
scoring_df

## Per-stratum breakdown

Strata are `(unit_type, gold_label)`: the sampling axes. Macro-averaging
per-flow F1 within each stratum shows where a model is strong or weak
(e.g. clean detection on `benign` units vs. recall on `malicious host_egress`).

In [ ]:
stratum = (
    units_df.assign(stratum=units_df["unit_type"] + " / " + units_df["gold_label"])
    .groupby(["stratum", "provider", "persona"])
    .agg(
        units=("eval_unit_id", "count"),
        per_flow_f1=("per_flow_f1", "mean"),
        per_pair_f1=("per_pair_f1", "mean"),
        per_host_f1=("per_host_f1", "mean"),
        first_pass_valid_rate=("unit_first_pass_valid", "mean"),
        defects=("defect", lambda s: s.notna().sum()),
    )
    .round(4)
)
stratum

## Cost, latency, and cache

Cost is rolled up per `provider/persona`; latency percentiles are per-call;
the cache block reports `cached_tokens` savings.

In [ ]:
cost_df = pd.DataFrame(summary["per_provider_persona"]).T
cost_df.index.name = "provider/persona"
display(cost_df)

print("latency per call (ms):")
display(pd.DataFrame(summary["latency_per_call_ms"]).T)

print("cache:")
print(json.dumps(summary["cache"], indent=2))

## Ablation deltas (if available)

If an `ablation_summary.json` exists for this run's `(dataset_hash, seed)`,
show the `tools_off → main` / `playbooks_off → main` deltas. Positive means
`main` scores higher, i.e. the ablated layer was contributing lift.

In [ ]:
abl_path = (
    REPO / "ablations" / metadata["dataset_hash"] / str(metadata["sample_seed"])
    / "ablation_summary.json"
)
if abl_path.exists():
    abl = json.loads(abl_path.read_text())
    print("runs joined:", abl["runs"])
    print("missing:", abl["missing_ablations"])
    for name, block in abl["deltas"].items():
        if block:
            print(f"\n=== {name} ===")
            display(pd.DataFrame(block).T.round(4))
else:
    print(f"no ablation_summary.json at {abl_path.relative_to(REPO)}")
    print("Run `socbench aggregate --dataset-hash "
          f"{metadata['dataset_hash']} --seed {metadata['sample_seed']}` "
          "after running at least a `main` + one ablation.")